# 02 - Exploratory Data Analysis (EDA)

Load the merged dataset, explore distributions, time-series patterns, correlations, and discuss data leakage risks.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_merged_data
from src.utils import get_data_dir

# Load data (use synthetic if merged_data.csv missing)
try:
    df = load_merged_data()
except FileNotFoundError:
    from src.synthetic_data import generate_synthetic_dataset
    df = generate_synthetic_dataset()
    get_data_dir().mkdir(parents=True, exist_ok=True)
    df.to_csv(get_data_dir() / 'merged_data.csv', index=False)

df['date'] = pd.to_datetime(df['date'])
print(df.shape)
df.head()

## 1. Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, col in zip(axes.flat, ['NDVI', 'EVI', 'yield', 'temperature', 'precipitation', 'soil_OC']):
    if col in df.columns:
        df[col].hist(ax=ax, bins=30)
        ax.set_title(col)
plt.tight_layout()
plt.savefig('../reports/figures/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. NDVI Time-Series for Sample Fields

In [ ]:
sample_fields = df['field_id'].unique()[:5]
fig, ax = plt.subplots(figsize=(10, 5))
for fid in sample_fields:
    grp = df[df['field_id'] == fid].sort_values('date')
    ax.plot(grp['date'], grp['NDVI'], label=fid, marker='o', markersize=4)
ax.set_xlabel('Date')
ax.set_ylabel('NDVI')
ax.set_title('NDVI Time-Series (Sample Fields)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/eda_ndvi_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Correlation Heatmap & NDVI–Yield Interaction

In [ ]:
agg = df.groupby('field_id').agg({
    'NDVI': 'max', 'EVI': 'max', 'yield': 'first',
    'precipitation': 'sum', 'temperature': 'mean', 'soil_OC': 'first'
}).reset_index()

plt.figure(figsize=(8, 6))
numeric = agg.select_dtypes(include=[np.number])
sns.heatmap(numeric.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap (Field-Level Aggregates)')
plt.tight_layout()
plt.savefig('../reports/figures/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Interaction: NDVI peak vs yield, stratified by precipitation
agg['precip_high'] = agg['precipitation'] > agg['precipitation'].median()
plt.figure(figsize=(8, 5))
for label, sub in agg.groupby('precip_high'):
    plt.scatter(sub['NDVI'], sub['yield'], label=f'Precip {"High" if label else "Low"}', alpha=0.7)
plt.xlabel('Peak NDVI')
plt.ylabel('Yield (t/ha)')
plt.title('NDVI–Yield: Precipitation Modulates Relationship')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/eda_ndvi_yield_interaction.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Missing Data & Outliers

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nOutliers (IQR):')
for col in ['NDVI', 'yield']:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    print(f'  {col}: {n_out}')

## 5. Data Leakage Prevention

- **Target alignment**: Yield labels are for the same year as satellite/weather data.
- **Temporal split**: We use data from April–July (1 month before harvest) only.
- **No future weather**: Cumulative precip and GDD are computed up to prediction date only.